# FLEX-clash: Inner Product Manipulation (IPM) Attack

This notebook demonstrates how to execute the **Inner Product Manipulation (IPM)** model poisoning attack using `flexclash`.

### Background
The IPM attack was introduced by **Xie et al. (UAI 2020)** in [*Fall of Empires: Breaking Byzantine-tolerant SGD by Inner Product Manipulation*](https://proceedings.mlr.press/v115/xie20a.html).

In Byzantine federated learning, honest clients compute local model updates $\Delta w_i = w_i - w_{\text{global}}$. The IPM attack crafts Byzantine updates directed in the exact opposite direction to the average honest update:

$$v = -\epsilon \cdot \bar{\Delta w} = -\epsilon \cdot \frac{1}{|\mathcal{H}|} \sum_{i \in \mathcal{H}} (w_i - w_{\text{global}})$$

This guarantees a negative inner product with the true gradient direction:

$$\langle v, \bar{\Delta w} \rangle = -\epsilon \|\bar{\Delta w}\|^2 < 0$$

causing the global aggregation to steer away from the optimum.

## 1. Setting up the Federated Environment with FLEX

We create a federated dataset and initialize a client-server pool with a PyTorch classification model.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from flex.data import Dataset, FedDataDistribution
from flex.model import FlexModel
from flex.pool import FlexPool, init_server_model
from flex.pool.primitives_pt import deploy_server_model_pt, collect_clients_weights_pt, set_aggregated_weights_pt
from flex.pool.aggregators import fed_avg
from flexclash.model import inner_product_manipulation, ipm_poisoner
from flexclash.pool import median, multikrum, trimmed_mean, bulyan

# Generate synthetic classification dataset
np.random.seed(42)
torch.manual_seed(42)
X = np.random.randn(300, 10).astype(np.float32)
y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(np.int64)

central_dataset = Dataset.from_array(X, y)
fed_dataset = FedDataDistribution.iid_distribution(central_dataset, n_nodes=10)

# Define model initialization
@init_server_model
def build_server_model():
    fm = FlexModel()
    fm["model"] = nn.Sequential(
        nn.Linear(10, 16),
        nn.ReLU(),
        nn.Linear(16, 2)
    )
    return fm

pool = FlexPool.client_server_pool(fed_dataset, init_func=build_server_model)
server = pool.servers
clients = pool.clients
aggregator = pool.aggregators

print(f"Created pool with {len(server)} server and {len(clients)} clients.")


## 2. Deploy Server Model and Train Locally on Clients

We broadcast the server model to all clients, then each client trains on its local partition.

In [ ]:
# Deploy global model to clients
server.map(deploy_server_model_pt, clients)

# Local training function
def local_train(client_model, client_dataset):
    model = client_model["model"]
    optimizer = optim.SGD(model.parameters(), lr=0.05)
    criterion = nn.CrossEntropyLoss()
    model.train()
    
    X_t = torch.tensor(np.array(client_dataset.X_data), dtype=torch.float32)
    y_t = torch.tensor(np.array(client_dataset.y_data), dtype=torch.long)
    
    for epoch in range(10):
        optimizer.zero_grad()
        loss = criterion(model(X_t), y_t)
        loss.backward()
        optimizer.step()

# Run training across all clients
clients.map(local_train)
print("Local training completed for all clients.")


## 3. Executing the IPM Attack

Suppose 2 out of 10 clients are Byzantine adversaries. We can execute IPM using:
1. High-level pool operator: `inner_product_manipulation(clients, server_model, malicious_clients=[...], epsilon=1.0)`
2. Or `.map()` with `ipm_poisoner(server_model, honest_clients, epsilon=1.0)`

In [ ]:
server_model = server._models[list(server.actor_ids)[0]]
malicious_client_ids = list(clients.actor_ids)[:2]
honest_client_ids = list(clients.actor_ids)[2:]

print(f"Malicious clients ({len(malicious_client_ids)}): {malicious_client_ids}")
print(f"Honest clients ({len(honest_client_ids)}): {honest_client_ids}")

# Execute Inner Product Manipulation attack with epsilon=1.0
inner_product_manipulation(
    clients,
    server_model=server_model,
    malicious_clients=malicious_client_ids,
    epsilon=1.0
)
print("IPM attack applied to malicious clients!")


### Verifying the IPM Property: Negative Inner Product

Let us compute the inner product between the malicious update and the mean benign update vector:

In [ ]:
srv_weights = [p.data.clone() for p in server_model["model"].parameters()]

# Honest updates delta_h
h_updates = []
for hid in honest_client_ids:
    client_w = [p.data.clone() for p in clients._models[hid]["model"].parameters()]
    h_updates.append([cw - sw for cw, sw in zip(client_w, srv_weights)])

# Mean honest update
mean_h_update = [
    torch.stack([u[layer] for u in h_updates]).mean(dim=0)
    for layer in range(len(srv_weights))
]

# Malicious update
mal_w = [p.data.clone() for p in clients._models[malicious_client_ids[0]]["model"].parameters()]
mal_update = [mw - sw for mw, sw in zip(mal_w, srv_weights)]

# Inner product <v, delta_bar>
inner_prod = sum(torch.sum(mu * hu).item() for mu, hu in zip(mal_update, mean_h_update))
norm_sq = sum(torch.sum(hu ** 2).item() for hu in mean_h_update)

print(f"Mean benign update norm squared ||Δw_bar||^2: {norm_sq:.6f}")
print(f"Inner product <v_mal, Δw_bar>: {inner_prod:.6f}")
print(f"Expected (-ε * ||Δw_bar||^2): {-1.0 * norm_sq:.6f}")
assert inner_prod < 0, "Inner product should be negative!"
print("
Confirmed: Inner product is strictly negative, proving IPM manipulation!")


## 4. Aggregation and Byzantine Robust Defenses

Now the aggregator collects weights and we can evaluate different aggregation rules:
- Standard **FedAvg**
- Robust defenses: **MultiKrum**, **Bulyan**, **Trimmed Mean**, **Median**

In [ ]:
# Collect client weights into aggregator
aggregator.map(collect_clients_weights_pt, clients)

# Defenses in flexclash.pool
# We can defend using MultiKrum, Bulyan, Median, or Trimmed Mean:
aggregator.map(multikrum, f=2, m=6)
aggregator.map(set_aggregated_weights_pt, server)

print("MultiKrum aggregation completed and weights deployed to server.")
